# Partie II — Réseaux de Neurones Convolutifs (CNN) pour la Classification d'Images

L'objectif de ce notebook est de comprendre les CNN, leur supériorité sur les MLP pour les images, ainsi que les mécanismes fondamentaux de convolution, de pooling et de partage de poids. Nous allons implémenter ces opérations manuellement, puis les comparer aux implémentations PyTorch, et enfin entraîner un CNN complet sur le dataset CIFAR-10.

## 1. Théorie : Pourquoi le MLP Échoue sur les Images

Les réseaux de neurones entièrement connectés (MLP) présentent deux limitations majeures lorsqu'ils sont appliqués directement à des images :

- **Problème 1 — Explosion des paramètres** : une image CIFAR-10 (32×32×3 = 3072 pixels) avec un MLP de 2 couches cachées de 512 neurones nécessite 3072×512 + 512×512 = ~1.8M paramètres rien que pour les premières couches. Cela est très inefficace et rend l'entraînement difficile.

- **Problème 2 — Absence d'invariance spatiale** : un MLP ne « sait » pas que des pixels voisins forment des structures locales (bords, textures). Déplacer un objet de quelques pixels produit un vecteur d'entrée complètement différent, ce qui oblige le réseau à réapprendre les mêmes patterns en toutes positions.

- **Solution CNN — 3 principes fondamentaux** :
  1. **Localité** : chaque neurone ne voit qu'une région locale de l'image, appelée champ récepteur (*receptive field*). Cela réduit drastiquement le nombre de connexions.
  2. **Partage de poids** : le même filtre (noyau de convolution) est appliqué à toute l'image. Ainsi, un détecteur de bords appris à un endroit peut être utilisé partout → des milliers de fois moins de paramètres.
  3. **Hiérarchie** : les couches successives apprennent des représentations de plus en plus abstraites : bords simples → formes géométriques → parties d'objets → objets complets.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torchvision
import torchvision.transforms as T

from deep_learning_project.utils import set_seed, get_device

set_seed(42)
device = get_device()
print(f"Device : {device}")

# Normalisation CIFAR-10 (mean/std calculés sur le dataset d'entraînement)
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

transform_train = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Téléchargement automatique dans data/
train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True,
                                              download=True, transform=transform_train)
test_dataset  = torchvision.datasets.CIFAR10(root='../data', train=False,
                                              download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128,
                                            shuffle=True, num_workers=2, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=128,
                                            shuffle=False, num_workers=2, pin_memory=True)

CLASSES = train_dataset.classes
print(f"Taille entraînement : {len(train_dataset)}, test : {len(test_dataset)}")
print(f"Classes : {CLASSES}")

In [ ]:
# Afficher quelques images CIFAR-10
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
# Use raw dataset (no normalization) for display
raw_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=False,
                                            transform=T.ToTensor())
for idx, ax in enumerate(axes.flat):
    img, label = raw_dataset[idx]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[label], fontsize=8)
    ax.axis('off')
plt.suptitle("Exemples CIFAR-10", fontsize=12)
plt.tight_layout()
from deep_learning_project.utils.io import save_figure
save_figure(fig, "cifar10_samples.png")
plt.show()

## 2. Théorie : La Convolution 2D

L'opération centrale d'un CNN est la **cross-corrélation 2D** (que PyTorch appelle par abus de langage « convolution »). Elle consiste à faire glisser un petit noyau $K$ sur l'image $I$ et à calculer le produit scalaire local :

$$
(I \star K)[i,j] = \sum_m \sum_n I[i+m,\, j+n] \cdot K[m,n]
$$

> **Note :** PyTorch appelle « convolution » ce qui est techniquement une cross-corrélation (pas de retournement du noyau). En pratique, la distinction est sans importance car les poids sont appris.

Les principaux **hyperparamètres** d'une couche convolutive sont :
- **kernel_size** $k$ : taille du noyau (ex. 3×3, 5×5)
- **padding** $p$ : nombre de zéros ajoutés en bordure pour contrôler la taille de sortie
- **stride** $s$ : pas de déplacement du noyau (stride=2 divise la résolution par 2)

La **dimension de sortie** se calcule par la formule :

$$
H_{\text{out}} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1
$$

In [ ]:
from deep_learning_project.models.cnn_manual import (
    cross_correlate_2d, cross_correlate_2d_batch,
    max_pool_2d, avg_pool_2d, output_size_formula
)

# Démonstration avec un noyau de détection de bords (Sobel)
sobel_x = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]], dtype=np.float32)

# Prendre une image CIFAR-10 en niveaux de gris
raw_img, label = raw_dataset[0]
gray = raw_img.mean(0).numpy()  # (32, 32)

# Appliquer la cross-corrélation manuelle
edge_map = cross_correlate_2d(gray, sobel_x, padding=1, stride=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title(f"Image originale ({CLASSES[label]})")
axes[0].axis('off')
axes[1].imshow(sobel_x, cmap='RdBu')
axes[1].set_title("Noyau Sobel-X (détection bords verticaux)")
axes[2].imshow(np.abs(edge_map), cmap='hot')
axes[2].set_title(f"Résultat : {edge_map.shape}")
axes[2].axis('off')
plt.suptitle("Démonstration de la Cross-Corrélation Manuelle", fontsize=12)
plt.tight_layout()
save_figure(fig, "cnn_manual_convolution_demo.png")
plt.show()

print("Vérification avec PyTorch nn.Conv2d :")
conv_pt = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    conv_pt.weight.copy_(torch.tensor(sobel_x).unsqueeze(0).unsqueeze(0))
input_pt = torch.tensor(gray).unsqueeze(0).unsqueeze(0)
edge_pt = conv_pt(input_pt).squeeze().detach().numpy()
print(f"  Erreur max (manuelle vs PyTorch) : {np.max(np.abs(edge_map - edge_pt)):.6f}")

## 3. Calcul des Dimensions

Comprendre comment les dimensions évoluent à travers les couches est **indispensable** avant de construire une architecture CNN. Une erreur de dimension est l'une des causes les plus fréquentes d'erreur lors de l'implémentation.

Le tableau ci-dessous illustre l'effet des différents paramètres sur la dimension de sortie, en utilisant la formule :

$$H_{\text{out}} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

In [ ]:
print("=== Formule : out = floor((H + 2p - k) / s) + 1 ===\n")
print(f"{'H':>5} {'k':>5} {'p':>5} {'s':>5} {'out':>5}")
print("-" * 30)
configs = [
    (32, 5, 0, 1), (32, 5, 2, 1), (32, 3, 1, 1),
    (28, 5, 0, 1), (14, 3, 0, 2), (14, 2, 0, 2),
]
for H, k, p, s in configs:
    out = output_size_formula(H, k, p, s)
    print(f"{H:>5} {k:>5} {p:>5} {s:>5} {out:>5}")

print("\n=== Dimensions pour LeNet sur CIFAR-10 (32x32) ===")
print("Couche               | Entrée    | Noyau | P | S | Sortie")
print("-" * 65)
H = 32
for desc, k, p, s in [("Conv1 (6 filtres)", 5, 0, 1), ("MaxPool1", 2, 0, 2),
                        ("Conv2 (16 filtres)", 5, 0, 1), ("MaxPool2", 2, 0, 2)]:
    out = output_size_formula(H, k, p, s)
    print(f"{desc:20s} | {H:>3}x{H:>3}    | {k:>5} | {p} | {s} | {out}x{out}")
    H = out
print(f"\nDimension aplatie : 16 × {H} × {H} = {16*H*H}")

## 4. Théorie : Le Pooling

Le **pooling** est une opération de sous-échantillonnage qui réduit la résolution spatiale des feature maps. Elle apporte deux avantages : réduire le nombre de paramètres dans les couches suivantes et introduire une invariance locale aux petites translations.

Il existe deux variantes principales :

- **Max Pooling** : retient la valeur maximale dans chaque fenêtre. Il conserve l'activation la plus forte, c'est-à-dire la feature la plus présente (dominant feature). C'est la variante la plus utilisée en pratique pour la classification, car elle est robuste au bruit et met en valeur les détections nettes.

- **Average Pooling** : retient la moyenne des valeurs dans chaque fenêtre. Il produit un résumé général de la région, moins sensible aux valeurs extrêmes. Utilisé notamment dans les architectures modernes (ex. Global Average Pooling en fin de réseau, à la place d'une couche fully-connected).

En pratique, le max pooling est préféré pour les couches intermédiaires d'un CNN de classification car il préserve mieux les activations fortes.

In [ ]:
# Démonstration du pooling manuel
feature_map = gray[np.newaxis, :, :]  # (1, 32, 32)

pooled_max = max_pool_2d(feature_map, pool_size=2, stride=2)
pooled_avg = avg_pool_2d(feature_map, pool_size=2, stride=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(feature_map[0], cmap='gray')
axes[0].set_title(f"Original : {feature_map.shape[1]}×{feature_map.shape[2]}")
axes[1].imshow(pooled_max[0], cmap='gray')
axes[1].set_title(f"Max Pooling 2×2 : {pooled_max.shape[1]}×{pooled_max.shape[2]}")
axes[2].imshow(pooled_avg[0], cmap='gray')
axes[2].set_title(f"Avg Pooling 2×2 : {pooled_avg.shape[1]}×{pooled_avg.shape[2]}")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
save_figure(fig, "pooling_comparison.png")
plt.show()

## 5. Modèle CNN : Architecture LeNet-Style

Nous utilisons la classe `LeNetCNN`, une architecture inspirée du LeNet-5 originel (LeCun et al., 1998) mais adaptée à CIFAR-10 (images couleur 32×32). La configuration par défaut comprend :

- 2 blocs convolutifs (Conv → BN optionnel → ReLU → MaxPool)
- 2 couches fully-connected avec activation ReLU
- Une couche de sortie à 10 classes

Les paramètres configurables permettent d'explorer différents choix architecturaux : nombre de filtres, tailles des noyaux, padding, batch normalization et dropout.

In [ ]:
from deep_learning_project.models.cnn import LeNetCNN

model_cnn = LeNetCNN(
    in_channels=3,
    input_hw=(32, 32),
    num_classes=10,
    conv_channels=[6, 16],
    kernel_sizes=[5, 5],
    pool_sizes=[2, 2],
    paddings=[0, 0],
    fc_dims=[120, 84],
    activation="relu",
    dropout_p=0.0,
    batch_norm=False,
).to(device)

print(model_cnn)
total_params = sum(p.numel() for p in model_cnn.parameters())
trainable_params = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(f"\nParamètres totaux    : {total_params:,}")
print(f"Paramètres entraînables : {trainable_params:,}")

## 6. Expériences : Impact des Hyperparamètres

Pour comprendre l'influence des choix architecturaux, nous comparons plusieurs configurations de CNN entraînées sur un sous-ensemble de CIFAR-10 pendant 10 époques. Les configurations testées sont :

- **Baseline (LeNet)** : configuration originale avec 6 et 16 filtres, noyaux 5×5, sans padding
- **Padding=2** : ajout de padding pour conserver la résolution spatiale plus longtemps
- **Stride=2** : utilisation de stride pour le sous-échantillonnage à la place du pooling
- **+ Filtres [32,64]** : augmentation du nombre de filtres avec noyaux 3×3
- **+ BatchNorm** : ajout de la normalisation par lots pour une meilleure stabilité

In [ ]:
from deep_learning_project.training.train_mlp import train

def quick_train_cnn(config_name, conv_channels, kernel_sizes, pool_sizes, paddings,
                     fc_dims, batch_norm=False, n_epochs=10):
    """Entraîne un CNN pendant n_epochs et retourne la meilleure val_acc."""
    set_seed(42)
    model = LeNetCNN(
        in_channels=3, input_hw=(32, 32), num_classes=10,
        conv_channels=conv_channels, kernel_sizes=kernel_sizes,
        pool_sizes=pool_sizes, paddings=paddings, fc_dims=fc_dims,
        batch_norm=batch_norm,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # Use smaller dataset for quick experiments
    from torch.utils.data import DataLoader, Subset
    import random as rnd
    rnd.seed(42)
    train_sub = Subset(train_dataset, rnd.sample(range(len(train_dataset)), 5000))
    val_sub   = Subset(test_dataset,  rnd.sample(range(len(test_dataset)),  1000))
    quick_train = DataLoader(train_sub, batch_size=128, shuffle=True)
    quick_val   = DataLoader(val_sub,   batch_size=128, shuffle=False)

    history = train(model, quick_train, quick_val, optimizer, criterion,
                    device, epochs=n_epochs, verbose=False)
    n_params = sum(p.numel() for p in model.parameters())
    return {
        "Config": config_name,
        "Paramètres": f"{n_params:,}",
        "Best Val Acc": f"{max(history['val_acc']):.3f}",
    }, model, history

experiments = [
    ("Baseline (LeNet)",       [6,16],  [5,5], [2,2], [0,0], [120,84], False),
    ("Padding=2",              [6,16],  [5,5], [2,2], [2,2], [120,84], False),
    ("Stride=2",               [6,16],  [3,3], [None,None], [1,1], [120,84], False),
    ("+ Filtres [32,64]",      [32,64], [3,3], [2,2], [1,1], [256,128], False),
    ("+ BatchNorm",            [32,64], [3,3], [2,2], [1,1], [256,128], True),
]

results = []
best_acc = 0
best_model_cnn = None
best_history = None

for exp_name, cc, ks, ps, pad, fcd, bn in experiments:
    row, model, history = quick_train_cnn(exp_name, cc, ks, ps, pad, fcd, bn)
    results.append(row)
    acc = float(row["Best Val Acc"])
    print(f"  {exp_name:25s} | Acc={acc:.3f} | Params={row['Paramètres']}")
    if acc > best_acc:
        best_acc = acc
        best_model_cnn = model
        best_history = history
        best_config = exp_name

results_df = pd.DataFrame(results).sort_values("Best Val Acc", ascending=False)
print(f"\n=== Tableau des résultats (trié par accuracy) ===")
print(results_df.to_string(index=False))
print(f"\nMeilleure config : {best_config}")

## 7. Entraînement Complet du Meilleur Modèle

Nous entraînons maintenant le meilleur modèle (CNN avec 32 et 64 filtres, noyaux 3×3, batch normalization et dropout) sur l'intégralité du dataset CIFAR-10 pendant 20 époques. Un scheduler de taux d'apprentissage (StepLR) est utilisé pour réduire le lr de moitié toutes les 10 époques.

In [ ]:
from deep_learning_project.evaluation.classification import plot_training_history

set_seed(42)
final_cnn = LeNetCNN(
    in_channels=3, input_hw=(32, 32), num_classes=10,
    conv_channels=[32, 64], kernel_sizes=[3, 3],
    pool_sizes=[2, 2], paddings=[1, 1], fc_dims=[256, 128],
    batch_norm=True, dropout_p=0.3,
).to(device)

optimizer = torch.optim.Adam(final_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.CrossEntropyLoss()

# Partial training (20 epochs for demo — increase for better accuracy)
cnn_history = train(
    final_cnn, train_loader, test_loader, optimizer, criterion,
    device, epochs=20, checkpoint_filename="cnn_final_best.pt", verbose=True,
)

fig = plot_training_history(cnn_history, "CNN LeNet — Courbes d'Entraînement",
                             save_filename="cnn_training_curves.png")
plt.show()

## 8. Visualisation des Feature Maps

Une des grandes forces des CNN est leur interprétabilité partielle : on peut visualiser ce que chaque couche convolutive « voit ». Les **feature maps** sont les sorties des filtres pour une image donnée.

- **Première couche** : les filtres détectent des features bas niveau — bords, contrastes, textures simples. Ces représentations sont proches de la perception visuelle humaine.
- **Couches profondes** : les filtres détectent des features de plus en plus abstraites — formes, parties d'objets, concepts sémantiques.

Cette hiérarchie de représentations est le fondement de la puissance des CNN pour la vision par ordinateur.

In [ ]:
from deep_learning_project.utils.io import load_checkpoint
load_checkpoint(final_cnn, "cnn_final_best.pt", device)
final_cnn.eval()

# Prendre une image test
sample_img, sample_label = test_dataset[0]
sample_input = sample_img.unsqueeze(0).to(device)

# Obtenir les feature maps intermédiaires
with torch.no_grad():
    feature_maps = final_cnn.get_feature_maps(sample_input)

print(f"Image : {CLASSES[sample_label]}")
for i, fm in enumerate(feature_maps):
    print(f"  Couche conv {i+1} : {fm.shape}")  # (1, C, H, W)

# Visualiser la première couche (bords, textures)
fm1 = feature_maps[0].squeeze(0).cpu().numpy()  # (C, H, W)
n_channels = min(fm1.shape[0], 16)
cols = 8
rows = (n_channels + cols - 1) // cols

fig, axes = plt.subplots(rows + 1, cols, figsize=(16, 3*(rows+1)))
# Image originale (dénormalisée)
orig = sample_img.permute(1, 2, 0).numpy()
orig = orig * np.array(CIFAR_STD) + np.array(CIFAR_MEAN)
orig = np.clip(orig, 0, 1)
axes[0, 0].imshow(orig)
axes[0, 0].set_title(f"Entrée: {CLASSES[sample_label]}", fontsize=8)
for ax in axes[0, 1:]:
    ax.axis('off')
for i in range(n_channels):
    r, c = i // cols + 1, i % cols
    axes[r, c].imshow(fm1[i], cmap='viridis')
    axes[r, c].set_title(f"Filtre {i}", fontsize=7)
    axes[r, c].axis('off')
for i in range(n_channels, rows * cols):
    axes[i // cols + 1, i % cols].axis('off')

plt.suptitle("Feature Maps — Première Couche Convolutive", fontsize=12)
plt.tight_layout()
save_figure(fig, "cnn_feature_maps_layer1.png")
plt.show()

## 9. Comparaison MLP vs CNN

Pour mesurer concrètement la supériorité des CNN sur les MLP pour les images, nous entraînons un MLP classique sur les images CIFAR-10 aplaties (3072 dimensions d'entrée) et comparons :

- Le nombre de paramètres de chaque modèle
- L'accuracy de validation obtenue en 10 époques
- Les courbes d'apprentissage (loss et accuracy)

Le MLP traite chaque pixel indépendamment et ne peut pas exploiter les relations spatiales locales, ce qui le désavantage fortement par rapport au CNN.

In [ ]:
from deep_learning_project.models.mlp import MLP
from deep_learning_project.evaluation.classification import get_predictions, compute_accuracy

# MLP sur images aplaties (3072 entrées)
set_seed(42)
mlp_on_images = MLP(
    input_dim=3072,
    hidden_dims=[512, 256],
    output_dim=10,
    activation="relu",
    dropout_p=0.3,
).to(device)

# Adapter le DataLoader pour aplatir les images
class FlatDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return img.view(-1), label  # (3072,)

flat_train = torch.utils.data.DataLoader(FlatDataset(train_dataset), batch_size=128, shuffle=True)
flat_test  = torch.utils.data.DataLoader(FlatDataset(test_dataset),  batch_size=128, shuffle=False)

optimizer_mlp = torch.optim.Adam(mlp_on_images.parameters(), lr=1e-3)
mlp_history = train(mlp_on_images, flat_train, flat_test, optimizer_mlp,
                     criterion, device, epochs=10, verbose=False)

# Comparaison
cnn_params = sum(p.numel() for p in final_cnn.parameters())
mlp_params = sum(p.numel() for p in mlp_on_images.parameters())
cnn_test_acc = max(cnn_history['val_acc'])
mlp_test_acc = max(mlp_history['val_acc'])

print("=" * 55)
print(f"{'Modèle':15s} | {'Paramètres':>12s} | {'Val Acc (10 ep)':>15s}")
print("-" * 55)
print(f"{'MLP (aplati)':15s} | {mlp_params:>12,} | {mlp_test_acc:>15.4f}")
print(f"{'CNN LeNet':15s} | {cnn_params:>12,} | {cnn_test_acc:>15.4f}")
print("=" * 55)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(mlp_history["val_acc"], label="MLP", color="orange")
axes[0].plot(cnn_history["val_acc"], label="CNN", color="blue")
axes[0].set_title("Accuracy validation : MLP vs CNN")
axes[0].set_xlabel("Époque")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(mlp_history["val_loss"], label="MLP", color="orange")
axes[1].plot(cnn_history["val_loss"], label="CNN", color="blue")
axes[1].set_title("Loss validation : MLP vs CNN")
axes[1].set_xlabel("Époque")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, "mlp_vs_cnn_comparison.png")
plt.show()

## 10. Résultats et Interprétation

Les expériences menées dans ce notebook permettent de tirer plusieurs conclusions importantes :

- **CNN > MLP même avec moins de paramètres** : le partage de poids et la localité sont la clé. Le CNN exploite la structure spatiale de l'image, là où le MLP traite chaque pixel indépendamment.

- **Le batch normalization accélère l'entraînement et améliore la généralisation** : en normalisant les activations à chaque couche, il stabilise le gradient et permet d'utiliser des taux d'apprentissage plus élevés.

- **Les feature maps de la première couche montrent des détecteurs de bords et de textures** : on retrouve des patterns similaires aux filtres Sobel et Gabor, confirmant que le réseau apprend des représentations perceptuellement cohérentes.

- **Avec plus d'époques et des architectures plus profondes** (ResNet, VGG, DenseNet), on atteint facilement >90% sur CIFAR-10. Notre LeNet-style avec 20 époques donne un résultat raisonnable comme point de départ.

## 11. Conclusion

Dans ce notebook, nous avons couvert les concepts fondamentaux des réseaux de neurones convolutifs :

1. Les **limitations des MLP** pour les images (explosion de paramètres, absence d'invariance spatiale)
2. Le **mécanisme de cross-corrélation 2D** et son implémentation manuelle
3. Le **calcul des dimensions** à travers les couches convolutives
4. Le **pooling** (max et average) et son rôle dans la réduction spatiale
5. L'**architecture LeNet** et l'impact des hyperparamètres (filtres, padding, batch norm)
6. La **visualisation des feature maps** pour interpréter ce que le réseau apprend
7. La **comparaison quantitative** MLP vs CNN sur CIFAR-10

Le prochain notebook abordera les **réseaux récurrents (RNN, LSTM, GRU)** pour le traitement de données séquentielles — une autre famille d'architectures fondamentales du deep learning.